# Import Required Libraries for Data Processing and Modeling

This cell imports all necessary libraries for the workflow, including:

- **pandas** and **numpy** for data manipulation and numerical operations,
- **xgboost** for gradient boosting-based predictive modeling,
- **os** for file handling,
- **scikit-learn utilities** for dataset splitting and evaluation metrics
- **json** for handling structured data files.

These libraries support data preprocessing, feature engineering, model training, and performance evaluation.

In [1]:
import pandas as pd, numpy as np, xgboost as xgb, os
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
import json

ModuleNotFoundError: No module named 'xgboost'

# Define Model Features, Target Variable, and XGBoost Parameters

- **FEATURES**: List of input variables capturing spatial (source/destination), temporal (hour, peak indicators, cyclic encoding), seasonal, distance, speed, and zone-type characteristics.
- **TARGET**: The variable to be predicted, i.e., mean travel time in minutes.
- **XGB_PARAMS**: Hyperparameters for the XGBoost model, controlling tree complexity, learning rate, sampling, regularization, and reproducibility.

In [ ]:
FEATURES = ["source_id", "dest_id", "hod", "hod_sin", "hod_cos",
            "is_peak_morning", "is_peak_evening", "is_off_peak",
            "is_monsoon", "quarter", "distance_km", "speed_proxy",
            "src_it", "src_commercial", "src_residential", "src_industrial",
            "dst_it", "dst_commercial", "dst_residential", "dst_industrial",
]

TARGET = "mean_travel_time_min"

XGB_PARAMS = {"n_estimators":500, "max_depth":6, "learning_rate":0.05,
              "subsample":0.8, "colsample_bytree":0.8, "min_child_weight":5,
              "reg_alpha":0.1, "reg_lambda":1.0, "tree_method":"hist", "random_state":42
}

# Define Evaluation Metrics for Quantile Regression

- **Pinball Loss**: Measures the accuracy of quantile predictions, penalizing over- and under-estimation asymmetrically based on the quantile level \( \tau \).
- **PICP (Prediction Interval Coverage Probability)**: Computes the proportion of true values that fall within the predicted interval, indicating coverage reliability.
- **PINAW (Prediction Interval Normalized Average Width)**: Measures the average width of prediction intervals normalized by the data range, reflecting interval sharpness.

In [ ]:
# Pinball (quantile) loss function.
def pinball_loss(y_true, y_pred, tau):
    errors = y_true - y_pred
    return np.mean(np.where(errors >= 0, tau * errors, (tau - 1) * errors))

# Prediction Interval Coverage Probability.
def picp(y_true, y_lower, y_upper):
    return np.mean((y_true >= y_lower) & (y_true <= y_upper))

# Prediction Interval Normalized Average Width.
def pinaw(y_lower, y_upper, y_range):
    return np.mean(y_upper - y_lower) / y_range

# Train Quantile XGBoost Models and Evaluate Predictive Performance

This cell defines and runs the `train()` function to train quantile-based XGBoost models for travel time prediction.

The workflow includes:
- loading the final processed dataset,
- separating input features and target travel time,
- splitting the data into training, validation, and test sets using a 70:15:15 ratio,
- training three XGBoost quantile regression models for the 10th, 50th, and 90th percentiles,
- applying early stopping using the validation set,
- generating predictions on the test set,
- evaluating the median model using **MAE**, **RMSE**, and **MAPE**,
- assessing interval quality using **PICP** and **PINAW**,
- computing **pinball loss** for all three quantiles,
- extracting feature importance from the median model.

In [ ]:
def train():
    df = pd.read_csv("data_processed.csv")
    X = df[FEATURES]
    y = df[TARGET]
    
    # 70 / 15 / 15 split
    X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42)
    X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1765, random_state=42)
    print(f"  Train:{len(X_train):,} | Val:{len(X_val):,} | Test:{len(X_test):,}")
    
    models = {}
    preds_test = {}
    
    for q in [0.10, 0.50, 0.90]:
        print(f"  Training Q{int(q*100)} model...")
        model = xgb.XGBRegressor(objective="reg:quantileerror", quantile_alpha=q, early_stopping_rounds=50, **XGB_PARAMS)
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
        path = os.path.join("models", f"model_q{int(q*100)}.json")
        model.save_model(path)
        print(f"    Saved: {path} | Best iter: {model.best_iteration}")
        models[q] = model; preds_test[q] = model.predict(X_test)
        
    # Convert y_test once
    y_arr = y_test.values

    # Safe MAPE function to avoid division-by-zero issues
    def safe_mape(y_true, y_pred):
        y_true = np.array(y_true)
        y_pred = np.array(y_pred)
        mask = y_true != 0
        if mask.sum() == 0:
            return np.nan
        return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

    # Metrics for each quantile model
    mae_q10  = mean_absolute_error(y_arr, preds_test[0.10])
    rmse_q10 = np.sqrt(mean_squared_error(y_arr, preds_test[0.10]))
    mape_q10 = safe_mape(y_arr, preds_test[0.10])

    mae_q50  = mean_absolute_error(y_arr, preds_test[0.50])
    rmse_q50 = np.sqrt(mean_squared_error(y_arr, preds_test[0.50]))
    mape_q50 = safe_mape(y_arr, preds_test[0.50])

    mae_q90  = mean_absolute_error(y_arr, preds_test[0.90])
    rmse_q90 = np.sqrt(mean_squared_error(y_arr, preds_test[0.90]))
    mape_q90 = safe_mape(y_arr, preds_test[0.90])

    # Interval / quantile-specific metrics
    cov  = picp(y_arr, preds_test[0.10], preds_test[0.90])
    wid  = pinaw(preds_test[0.10], preds_test[0.90], y_arr.max()-y_arr.min())
    pb10 = pinball_loss(y_arr, preds_test[0.10], 0.10)
    pb50 = pinball_loss(y_arr, preds_test[0.50], 0.50)
    pb90 = pinball_loss(y_arr, preds_test[0.90], 0.90)
    
    print("\nTest Metrics:")
    print(f"  Q10 -> MAE:{mae_q10:.3f} min | RMSE:{rmse_q10:.3f} min | MAPE:{mape_q10:.2f}%")
    print(f"  Q50 -> MAE:{mae_q50:.3f} min | RMSE:{rmse_q50:.3f} min | MAPE:{mape_q50:.2f}%")
    print(f"  Q90 -> MAE:{mae_q90:.3f} min | RMSE:{rmse_q90:.3f} min | MAPE:{mape_q90:.2f}%")
    print(f"  PICP:{cov*100:.1f}% | PINAW:{wid:.4f}")
    print(f"  Pinball Q10:{pb10:.4f} | Q50:{pb50:.4f} | Q90:{pb90:.4f}")
    
    # Feature importance
    fi_q10 = pd.Series(models[0.10].feature_importances_, index=FEATURES).sort_values(ascending=False)
    print("  Top 5 features for Q10:", list(fi_q10.head(5).index))
    fi_q50 = pd.Series(models[0.50].feature_importances_, index=FEATURES).sort_values(ascending=False)
    print("  Top 5 features for Q50:", list(fi_q50.head(5).index))
    fi_q90 = pd.Series(models[0.90].feature_importances_, index=FEATURES).sort_values(ascending=False)
    print("  Top 5 features for Q90:", list(fi_q90.head(5).index))
    
    # Save metrics
    metrics = {"MAE_Q10_min": round(mae_q10, 3),
               "RMSE_Q10_min": round(rmse_q10, 3),
               "MAPE_Q10_pct": round(mape_q10, 2),
               
               "MAE_Q50_min": round(mae_q50, 3),
               "RMSE_Q50_min": round(rmse_q50, 3),
               "MAPE_Q50_pct": round(mape_q50, 2),
               
               "MAE_Q90_min": round(mae_q90, 3),
               "RMSE_Q90_min": round(rmse_q90, 3),
               "MAPE_Q90_pct": round(mape_q90, 2),
               
               "PICP_pct":round(cov*100,1),"PINAW":round(wid,4), 
               "PB_Q10":round(pb10,4), "PB_Q50":round(pb50,4), "PB_Q90":round(pb90,4)}
    
    pd.DataFrame([metrics]).to_csv("ml_metrics.csv",index=False)
    
train()

In [ ]:
met = pd.read_csv("ml_metrics.csv")
met

In [ ]:
ZONES = ["Whitefield","Koramangala","Indiranagar","Hebbal","Marathahalli",
         "Electronic City","Jayanagar","Rajajinagar","Yeshwanthpur","BTM Layout",
         "HSR Layout","Bannerghatta Rd","Yelahanka","Sarjapur Road","MG Road","Banashankari"]

ZTYPE = {"Whitefield":"IT","Koramangala":"Commercial","Indiranagar":"Commercial",
          "Hebbal":"Residential","Marathahalli":"IT","Electronic City":"IT", "Jayanagar":"Residential",
          "Rajajinagar":"Residential","Yeshwanthpur":"Industrial", "BTM Layout":"Residential",
          "HSR Layout":"Commercial","Bannerghatta Rd":"Residential","Yelahanka":"Residential",
          "Sarjapur Road":"IT","MG Road":"Commercial", "Banashankari":"Residential"}

# Generate Travel Time Matrices Using Trained Quantile Models

This cell defines and executes the `generate_matrices()` function to construct origin-destination travel time matrices for all zone pairs under specified temporal conditions.

The workflow includes:
- loading the distance matrix and trained quantile XGBoost models (Q10, Q50, Q90),
- generating feature vectors for all valid OD pairs (excluding self-loops),
- incorporating spatial, temporal, and zone-type features,
- predicting travel times for each quantile while enforcing a minimum threshold,
- constructing 16×16 matrices for Q10 (optimistic), Q50 (median), and Q90 (conservative) travel times,
- extracting empirical mean travel times from the dataset for comparison.

In [ ]:
def generate_matrices(hod, quarter):
    n = len(ZONES)
    dist_df = pd.read_csv("distance_matrix_osrm.csv", index_col=0)
    
    # Load trained models
    models = {}
    for qi in [10, 50, 90]:
        m = xgb.XGBRegressor()
        m.load_model(os.path.join("models", f"model_q{qi}.json"))
        models[qi] = m
        
    # Build feature rows for all 240 OD pairs
    rows = []
    for i in range(n):
        for j in range(n):
            if i == j: continue
            d  = dist_df.iloc[i, j]
            st = ZTYPE[ZONES[i]]; dt = ZTYPE[ZONES[j]]
            rows.append({
                "source_id":i, "dest_id":j, "hod":hod,
                "hod_sin":np.sin(2*np.pi*hod/24),
                "hod_cos":np.cos(2*np.pi*hod/24),
                "is_peak_morning":int(hod in [7,8,9,10]),
                "is_peak_evening":int(hod in [17,18,19,20]),
                "is_off_peak":int(hod not in [7,8,9,10,17,18,19,20]),
                "quarter":quarter, "is_monsoon":int(quarter in [2,3]), 
                "distance_km":d, "speed_proxy":d/0.55,
                "src_it":int(st=="IT"), "src_commercial":int(st=="Commercial"),
                "src_residential":int(st=="Residential"), "src_industrial":int(st=="Industrial"),
                "dst_it":int(dt=="IT"), "dst_commercial":int(dt=="Commercial"),
                "dst_residential":int(dt=="Residential"), "dst_industrial":int(dt=="Industrial")
            })
            
    X = pd.DataFrame(rows)[FEATURES]
    
    # Predict all quantiles
    preds = {}
    for qi, m in models.items():
        preds[qi] = np.maximum(m.predict(X), 1.0)
        
    # Build 16x16 matrices
    q10 = np.zeros((n,n)); q50 = np.zeros((n,n)); q90 = np.zeros((n,n))
    idx = 0
    for i in range(n):
        for j in range(n):
            if i == j: continue
            q10[i][j] = round(preds[10][idx], 2)
            q50[i][j] = round(preds[50][idx], 2)
            q90[i][j] = round(preds[90][idx], 2)
            idx += 1
            
    # Extract Uber mean travel times for reference
    df_uber = pd.read_csv("data_processed.csv")
    df_hod  = df_uber[df_uber["hod"] == hod]
    mean_m  = np.zeros((n,n))
    for i in range(n):
        for j in range(n):
            if i == j: continue
            m2 = df_hod[(df_hod["source_id"]==i) & (df_hod["dest_id"]==j)]
            mean_m[i][j] = round(m2["mean_travel_time_min"].mean() if len(m2) else q50[i][j], 2)
            
    out = {"hod":hod, "zones":ZONES, "Q10":q10.tolist(), "Q50":q50.tolist(),
           "Q90":q90.tolist(), "mean_uber":mean_m.tolist()}
    
    with open("travel_time_matrices.json", "w") as f: json.dump(out, f, indent=2)
    print(f"  Saved: {"travel_time_matrices.json"}")

In [ ]:
generate_matrices(9, 2)